In [0]:
from pyspark.sql.functions import col, when, trim, regexp_replace, concat_ws, split

In [0]:
df = spark.table("novacart_catalog.001_bronze.order_items")

df.display()

In [0]:

from pyspark.sql import functions as F

df = spark.table("novacart_catalog.001_bronze.order_items")

df = df.withColumn("order_item_id", F.trim(F.col("order_item_id"))) \
       .withColumn("order_id", F.trim(F.col("order_id"))) \
       .withColumn("product_id", F.trim(F.col("product_id")))

df.display()

In [0]:
null_values = ["", "null", "NULL", "\\N", "-", "?"]
df = df.withColumn(
    "product_id",
    F.when(
        (F.col("product_id").isin(null_values)) | (F.col("product_id").isNull()),
        "unknown"
    ).otherwise(F.col("product_id"))
)
df.display()

In [0]:
df = df.withColumn(
    "order_item_id",
    F.when(F.col("order_item_id").rlike("^ITEM[0-9]+$"), F.col("order_item_id"))
     .otherwise(None)
)
df.display()

In [0]:
df = df.withColumn(
    "order_id",
    F.when(F.col("order_id").rlike("^ORD[0-9]+$"), F.col("order_id"))
     .otherwise(None)
)
df.display()

In [0]:
df = df.withColumn(
    "quantity",
    F.when(F.col("quantity") > 0, F.col("quantity")).otherwise(None)
)

In [0]:
df = df.withColumn(
    "unit_price",
    F.when(F.col("unit_price") > 0, F.col("unit_price")).otherwise(None)
)
df.display()

In [0]:
df = df.withColumn(
    "valid_order_id",
    F.when(F.col("order_id").isNotNull(), True).otherwise(False)
).withColumn(
    "valid_product_id",
    F.when((F.col("product_id").isNotNull()) & (F.col("product_id") != "unknown"), True).otherwise(False)
)

df.display()

In [0]:
for col_name in df.columns:
    new_col_name = col_name[0].upper() + col_name[1:] if len(col_name) > 0 else col_name
    df = df.withColumnRenamed(col_name, new_col_name)

df.display()

In [0]:
df = df.dropDuplicates()

In [0]:
display(df)

In [0]:
df.write.format("delta") .mode("overwrite") .option("overwriteSchema", "true") .saveAsTable("novacart_catalog.002_silver.order_items")